# DealRoom v3.6 — LLM Training with GRPO

**Goal:** Fine-tune Qwen2.5-3B-Instruct on the DealRoom negotiation environment using Unsloth QLoRA + TRL GRPOTrainer.

**What this notebook does:**
1. Installs Unsloth, TRL, and dependencies
2. Loads Qwen2.5-3B-Instruct with 4-bit QLoRA (fits in 15GB T4)
3. Defines the reward function: parse LLM action → execute in environment → return reward
4. Runs GRPO training for N episodes on the `hostile_acquisition` scenario
5. Saves the fine-tuned model and generates training curves

**VRAM budget:** ~11.5GB used / 15GB available (BS=1, GA=4, gradient checkpointing)
**Estimated time:** ~20 min for 50 episodes on T4 GPU

**Training approach:** Each GRPO sample runs a full episode (up to 8 steps). The cumulative episode reward provides rich signal for advantage estimation.

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth trl bitsandbytes peft accelerate transformers scipy numpy matplotlib networkx

In [ ]:
# Cell 2: Clone or update DealRoom from GitHub
import subprocess
import sys
from pathlib import Path

DEALROOM_REPO = "https://github.com/akshaypulla/deal_room.git"
BRANCH = "deal_room_v3"
TARGET_DIR = Path("/content/deal_room")

if TARGET_DIR.exists():
    print(f"✓ DealRoom already at {TARGET_DIR}")
    try:
        subprocess.run(["git", "-C", str(TARGET_DIR), "fetch", "origin"], 
                      capture_output=True, timeout=30)
        subprocess.run(["git", "-C", str(TARGET_DIR), "checkout", BRANCH], 
                      capture_output=True, timeout=30)
        subprocess.run(["git", "-C", str(TARGET_DIR), "pull", "--ff-only"], 
                      capture_output=True, timeout=30)
        print(f"  Switched to branch '{BRANCH}' and pulled latest.")
    except Exception as e:
        print(f"  Git update failed (fine if already on branch): {e}")
else:
    print(f"Cloning {DEALROOM_REPO} branch '{BRANCH}' ...")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, DEALROOM_REPO, str(TARGET_DIR)],
                    capture_output=True, check=True)
    print(f"✓ Cloned to {TARGET_DIR}")

sys.path.insert(0, str(TARGET_DIR))
print(f"sys.path updated with {TARGET_DIR}")

In [ ]:
# Cell 3: Verify environment loads
from deal_room.environment.dealroom_v3 import DealRoomV3
from deal_room.environment.text_env import DealRoomTextEnv
from deal_room.environment.prompts import build_situation_prompt, parse_action_text

env = DealRoomV3()
obs = env.reset(seed=42, task_id='hostile_acquisition')
print(f"Environment loaded. Round: {obs.round_number}/{obs.max_rounds}")
print(f"Stakeholders: {list(obs.stakeholders.keys())}")
print(f"Active blockers: {obs.active_blockers}")
env.close()

In [ ]:
# Cell 4: Test prompt building and action parsing
env = DealRoomTextEnv(task_id='hostile_acquisition', seed=42)
prompt = env.reset()
print("=== Initial Situation Prompt ===")
print(prompt[:1000], '...')
env.close()

In [ ]:
# Cell 5: Test action parsing
test_outputs = [
    'send_document Finance roi_model Here is our ROI model with downside scenarios.',
    'direct_message Legal I want to address your compliance concerns directly.',
    'concession Finance liability_cap=1500000',
    'group_proposal I propose we move to final contract review with agreed terms.',
    'This is a random unparseable output.',
]

for out in test_outputs:
    action = parse_action_text(out)
    print(f"Input:  {out[:60]}")
    print(f"Parsed: action_type={action.action_type}, target={action.target}")
    print()

In [ ]:
# Cell 6: Test full episode with random policy
import random

SAMPLE_ACTIONS = [
    'send_document Finance roi_model Here is our ROI model with approval-friendly assumptions.',
    'send_document Legal dpa Here is our DPA with enhanced liability safeguards.',
    'direct_message TechLead Can you share your technical requirements?',
    'direct_message Finance I understand your cost concerns.',
    'group_proposal I propose we move to final contract review.',
    'concession Operations timeline_weeks=12',
]

random.seed(42)
env = DealRoomTextEnv(task_id='hostile_acquisition', seed=42)
prompt = env.reset()

total_reward = 0
for step in range(8):
    action = random.choice(SAMPLE_ACTIONS)
    prompt, reward, done, info = env.step(action)
    total_reward += reward
    print(f"Step {step+1}: reward={reward:.3f}, done={done}, terminal={info.get('terminal_outcome', '?')}")
    if done:
        break

print(f"\nTotal episode reward: {total_reward:.3f}")
print(f"Final Pareto efficiency: {info.get('pareto_efficiency', 0):.3f}")
env.close()

In [ ]:
# Cell 7: Load Qwen2.5-3B with Unsloth 4-bit QLoRA
import torch
from unsloth import FastLanguageModel

model_name = "Qwen/Qwen2.5-3B-Instruct"
max_seq_length = 2048
dtype = torch.float16
load_in_4bit = True

print("Loading model with Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print(f"Model loaded. Vocab size: {tokenizer.vocab_size}")

In [ ]:
# Cell 8: Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=42,
)
print(f"LoRA adapters added. Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Cell 9: Define reward function for TRL
#
# Each completion is a SINGLE action evaluated in a FRESH environment.
# The reward is positive for valid send_document actions (~+0.75),
# lower for other action types (~+0.42), and negative for 
# veto-triggering or invalid outputs.
#
# To encourage concise output (stop at ~20-30 tokens):
# - Truncated/rambling outputs get a bonus penalty
# - Valid short send_document gets the highest reward

from typing import List
from deal_room.environment.text_env import DealRoomTextEnv
from deal_room.environment.prompts import parse_action_text

class DealRoomRewardFunction:
    """
    TRL-compatible single-step reward function.
    
    Each (prompt, completion) = ONE action in a fresh environment.
    Added: length penalty encourages concise output that stops naturally.
    """
    
    def __init__(self, task_id: str = "hostile_acquisition", base_seed: int = 42):
        self.task_id = task_id
        self.base_seed = base_seed
    
    def __call__(self, prompts: List[str], completions: List[str], completion_ids=None, **kwargs) -> List[float]:
        rewards = []
        for i, (prompt, completion) in enumerate(zip(prompts, completions)):
            seed = self.base_seed + i
            
            env = DealRoomTextEnv(task_id=self.task_id, seed=seed, use_llm_stakeholders=False)
            env.reset()
            
            _, reward, done, info = env.step(completion)
            
            # Length penalty: bonus for short, valid send_document
            # This encourages model to stop after action (not ramble)
            completion_len = len(completion.strip())
            
            action = parse_action_text(completion)
            
            if action and action.action_type == "send_document":
                # Positive reward for send_document actions
                if completion_len < 60:
                    # Short and valid: highest reward
                    reward = max(reward, 0.75)
                elif completion_len > 150:
                    # Too long: apply penalty (truncated/rambling)
                    reward -= 0.5
            elif action is None:
                # Parse failure: penalize heavily
                reward = -1.0
            
            env.close()
            rewards.append(float(reward))
        
        return rewards

reward_fn = DealRoomRewardFunction(task_id="hostile_acquisition", base_seed=42)
reward_fn.__name__ = "DealRoomRewardFunction"
print("Reward function: single-step + length penalty for concise output.")


In [ ]:
# Cell 10.5: Create training dataset with REAL DealRoom situation prompts
# Uses the actual build_situation_prompt() so training matches evaluation format.
# Each prompt is a different negotiation scenario (different seeds).
from datasets import Dataset
from deal_room.environment.text_env import DealRoomTextEnv
import sys

def generate_real_prompts(task_id="hostile_acquisition", num_prompts=50):
    """
    Generate diverse real prompts by running the environment with different seeds.
    Uses build_situation_prompt() so training = evaluation format.
    """
    prompts = []
    for seed in range(42, 42 + num_prompts):
        env = DealRoomTextEnv(task_id=task_id, seed=seed, use_llm_stakeholders=False)
        prompt = env.reset()
        env.close()
        prompts.append(prompt)
    return prompts

prompts = generate_real_prompts(num_prompts=50)
print(f"Dataset: {len(prompts)} prompts")
print(f"Prompt length: {len(prompts[0])} chars")
print(f"Sample (first 200 chars): {prompts[0][:200]}...")
train_dataset = Dataset.from_dict({"prompt": prompts})


In [ ]:
# Cell 10: Configure GRPO training
from trl import GRPOTrainer, GRPOConfig

training_args = GRPOConfig(
    output_dir="./dealroom_grpo_checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_prompt_length=1024,
    max_completion_length=256,
    gradient_checkpointing=True,
    logging_steps=5,
    save_steps=25,
    save_total_limit=3,
    max_steps=50,
    temperature=0.8,
    num_generations=8,
    epsilon=0.2,
    report_to="none",
    seed=42,
)
print("GRPOConfig initialized.")
print(f"Effective BS: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Samples per step: {training_args.num_generations}")

In [ ]:
# Cell 11: Initialize GRPOTrainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
print("GRPOTrainer initialized.")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
# Cell 12: Run GRPO training!
# Estimated: ~15-20 min on T4 for 50 steps
print("Starting GRPO training...")
print("=" * 60)
trainer.train()
print("=" * 60)
print("Training complete!")

In [ ]:
# Cell 13: Save fine-tuned model
output_dir = "dealroom-qwen-3b-negotiation"
model.save_pretrained_merged(output_dir, tokenizer, save_method="merged_16bit")
print(f"Model saved to: {output_dir}")

In [ ]:
# Cell 14: Generate training curves
import matplotlib.pyplot as plt
import numpy as np

log_history = trainer.state.log_history

steps = []
rewards = []
for entry in log_history:
    if "reward" in entry:
        steps.append(entry.get("step", 0))
        rewards.append(entry["reward"])

if steps:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, rewards, marker='o', linewidth=2, color='steelblue')
    
    # Moving average
    window = min(5, len(rewards))
    if window > 1:
        ma = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ma_steps = steps[:len(ma)]
        plt.plot(ma_steps, ma, linewidth=2, color='green', label=f'{window}-step MA')
        plt.legend()
    
    plt.title("DealRoom v3.6 — GRPO Training: Cumulative Episode Reward")
    plt.xlabel("Training Step")
    plt.ylabel("Cumulative Reward")
    plt.grid(True, alpha=0.3)
    plt.savefig("training_reward_curve.png", dpi=150, facecolor='white')
    plt.show()
    
    print(f"Saved to training_reward_curve.png")
    print(f"Initial: {rewards[0]:.3f}, Final: {rewards[-1]:.3f}, Change: {rewards[-1]-rewards[0]:+.3f}")
else:
    print("No reward data. Check TRL logs.")
    print("Keys in log_history[0]:", list(log_history[0].keys()) if log_history else "empty")

In [ ]:
# Cell 15: Evaluation episode with trained model
FastLanguageModel.for_inference(model)

env = DealRoomTextEnv(task_id='hostile_acquisition', seed=7)
prompt = env.reset()

print("=== Trained Model: Hostile Acquisition (seed=7) ===\n")
total_reward = 0
for step in range(8):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        temperature=0.8,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], 
        skip_special_tokens=True
    ).strip()
    
    print(f"Step {step+1}: {response[:80]}")
    prompt, reward, done, info = env.step(response)
    total_reward += reward
    print(f"         Reward: {reward:+.3f} | Done: {done}")
    if done:
        print(f"  -> {info.get('terminal_outcome', '?')}, Pareto: {info.get('pareto_efficiency', 0):.3f}")
        break
    print()

print(f"\nTotal reward: {total_reward:+.3f}")
env.close()

## Results Summary

After training, you should see:
1. **Reward curve** — cumulative reward over training steps
2. **Evaluation episode** — the trained LLM navigating a hostile scenario

### Next Steps:
- Push model to HuggingFace Hub: `model.push_to_hub_merged("your-username/dealroom-qwen-3b", tokenizer)`
- Use `06_inference_demo.ipynb` for interactive Gradio demo
- Record backup video of trained agent